# Aula 07: CRUD Avançado, Atualizações Atômicas e Indexação de Alta Performance no MongoDB
**Disciplina:** Banco de Dados Não-Relacionais (ADS16)  
**Curso:** Tecnologia em Análise e Desenvolvimento de Sistemas (ADS) — IFCE Campus Tauá  
**Professor:** Prof. Me. Reginaldo Fernandes  

---

## 🎯 Objetivos do Notebook:
1. Demonstrar a redução assintótica de $\mathcal{O}(N)$ para $\mathcal{O}(\log N)$ propiciada pela estrutura $B\text{-Tree}$.
2. Praticar filtros complexos com `$elemMatch` (resolvendo falsos positivos) e `$expr`.
3. Executar atualizações atômicas concorrentes (`$inc`, `$push` com `$slice`) e o operador `$[<elem>]` com `arrayFilters`.
4. Criar e gerenciar índices simples, compostos (Regra ESR), multikey e textuais com pesos.
5. Analisar planos de execução com `explain('executionStats')`, construir **Covered Queries** e visualizar benchmarks.

### 1. Configuração do Ambiente e Conexão

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from pymongo import MongoClient, ASCENDING, DESCENDING, TEXT

# Configuração visual acessível IFCE Tauá
plt.style.use('seaborn-v0_8-colorblind')
plt.rcParams.update({
    'figure.figsize': (11, 6),
    'axes.labelsize': 13,
    'axes.titlesize': 15,
    'lines.linewidth': 2.2
})

def despine(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=2000)
db = client["ecommerce_shopgrids"]
print("✅ Ambiente configurado e conectado ao banco 'ecommerce_shopgrids'!")

### 2. Fundamentos da $B\text{-Tree}$: Simulação Pareada $\mathcal{O}(N)$ vs $\mathcal{O}(\log N)$
Abaixo simulamos o custo de busca sequencial contra busca balanceada em memória para $N = 100.000$ itens:

In [2]:
import bisect

N = 100_000
dados_lineares = list(range(N))
dados_ordenados = sorted(dados_lineares)
alvo = 99_999 # Pior caso

# 1. Simulação COLLSCAN (O(N))
t0 = time.perf_counter()
for item in dados_lineares:
    if item == alvo:
        break
tempo_scan = (time.perf_counter() - t0) * 1000

# 2. Simulação IXSCAN B-Tree (O(log N))
t0 = time.perf_counter()
idx = bisect.bisect_left(dados_ordenados, alvo)
tempo_btree = (time.perf_counter() - t0) * 1000

print(f"Busca Sequencial (COLLSCAN): {tempo_scan:.4f} ms")
print(f"Busca B-Tree (IXSCAN)     : {tempo_btree:.4f} ms")
print(f"Eficiência: {tempo_scan / max(tempo_btree, 0.0001):.1f}x mais rápida!")

### 3. O Operador `$elemMatch` em Arrays de Subdocumentos
Demonstração ao vivo da resolução de falsos positivos em coleções de pedidos:

In [3]:
# Criando documento para teste
db.pedidos.delete_one({"codigo": "PED-NB-TESTE"})
db.pedidos.insert_one({
    "codigo": "PED-NB-TESTE",
    "itens": [
        {"sku": "TECLADO-PRO", "preco": 200.0, "qtd": 1},
        {"sku": "PILHA-AA", "preco": 10.0, "qtd": 6}
    ]
})

# Teste 1: Consulta Ingênua (Falso Positivo)
q_ingenuo = {"codigo": "PED-NB-TESTE", "itens.preco": {"$gt": 150}, "itens.qtd": {"$gte": 5}}
res_ingenuo = db.pedidos.find_one(q_ingenuo)
print("Consulta sem $elemMatch retornou documento?", "SIM (Falso Positivo!)" if res_ingenuo else "NÃO")

# Teste 2: Consulta Correta com $elemMatch
q_correto = {"codigo": "PED-NB-TESTE", "itens": {"$elemMatch": {"preco": {"$gt": 150}, "qtd": {"$gte": 5}}}}
res_correto = db.pedidos.find_one(q_correto)
print("Consulta com $elemMatch retornou documento?", "SIM" if res_correto else "NÃO (Correto!)")

### 4. Atualizações Atômicas: `$inc`, `$push` com `$slice` e `arrayFilters`

In [4]:
# 1. $inc de estoque e $push com $slice nos produtos
prod = db.produtos.find_one({"status": "ATIVO"})
if prod:
    sku = prod["sku"]
    db.produtos.update_one(
        {"sku": sku},
        {
            "$inc": {"estoque.atual": -1},
            "$push": {
                "ultimas_avaliacoes": {
                    "$each": [{"nota": 5, "autor": "Lab Notebook", "comentario": "Testado e aprovado!"}],
                    "$slice": -5
                }
            }
        }
    )
    atualizado = db.produtos.find_one({"sku": sku})
    print(f"Produto {sku} atualizado com sucesso. Estoque atual: {atualizado['estoque']['atual']}")

### 5. Criação de Índices, Regra ESR e Busca Textual

In [5]:
# 1. Índice Composto com Regra ESR (Equality, Sort, Range)
db.produtos.create_index(
    [("status", ASCENDING), ("data_cadastro", DESCENDING), ("preco", ASCENDING)],
    name="idx_esr_notebook"
)

# 2. Índice Textual Ponderado
db.produtos.create_index(
    [("titulo", TEXT), ("tags", TEXT), ("descricao", TEXT)],
    weights={"titulo": 10, "tags": 5, "descricao": 2},
    default_language="portuguese",
    name="idx_text_notebook"
)

# Consulta Textual com Ranking de Relevância
busca = db.produtos.find(
    {"$text": {"$search": "ergonômico sem fio"}},
    {"titulo": 1, "preco": 1, "score": {"$meta": "textScore"}, "_id": 0}
).sort([("score", {"$meta": "textScore"})]).limit(3)

print("Top 3 Produtos mais Relevantes:")
for item in busca:
    print(f"  Score: {item.get('score', 0):.2f} | R$ {item['preco']:>7.2f} | {item['titulo']}")

### 6. Diagnóstico com `explain()` e Covered Queries

In [6]:
# Índice de Cobertura Total
db.produtos.create_index(
    [("categoria", ASCENDING), ("preco", ASCENDING), ("sku", ASCENDING)],
    name="idx_cobertura_nb"
)

# Consulta Coberta (Covered Query): _id: 0 elimina o FETCH
exp = db.produtos.find(
    {"categoria": "Informática", "preco": {"$lte": 1500}},
    {"categoria": 1, "preco": 1, "sku": 1, "_id": 0}
).limit(10).explain("executionStats")

stats = exp["executionStats"]
print("Total de Documentos Físicos Examinados:", stats["totalDocsExamined"], "(0 = Covered Query!)")
print("Estágio de Execução:", stats["executionStages"]["stage"])
print("Tempo de Execução:", stats["executionTimeMillis"], "ms")